# Lab 75: The Minimal Group

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 8, *The Social Network*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-75.ipynb)

**What you will do:** tally how many tokens people allocate to their own arbitrary team versus the other one, compare the in-group share with an even 50/50 split, and ask a small AI chatbot to role-play a team member allocating tokens to check for the same bias.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 75 you split a group into two teams using a completely arbitrary criterion (a coin
flip, birthday month, or similar), and each anonymous participant allocated tokens between a
member of their own team and a member of the other team. This is the **minimal group paradigm**
(Tajfel et al., 1971): the book reports that most participants favour their own team, even
though the grouping was meaningless, the individuals were anonymous, and favouritism had no
strategic advantage.

## Record your results

Log each anonymous participant's token allocation between their own team and the other team.

In [ ]:
# example data: replace with yours
allocations = pd.DataFrame({
    "id": ["P01", "P02", "P03", "P04", "P05"],
    "tokens_to_ingroup": [7, 6, 8, 5, 9],
    "tokens_to_outgroup": [3, 4, 2, 5, 1],
})
display(allocations)
allocations[["tokens_to_ingroup", "tokens_to_outgroup"]].mean().plot.bar(
    color=["steelblue", "grey"], figsize=(5, 3.5))
plt.ylabel("mean tokens allocated"); plt.xticks(rotation=0); plt.show()

## Compare

Lab 75 does not give an exact percentage, only that "most participants allocate more tokens
to their own team." A 50/50 split is the natural no-bias baseline to check that claim against.

In [ ]:
ingroup_share = allocations.tokens_to_ingroup / (allocations.tokens_to_ingroup + allocations.tokens_to_outgroup)
mean_share = ingroup_share.mean() * 100
print(f"Mean share of tokens given to the in-group: {mean_share:.0f}% (50% would mean no bias)")
if mean_share > 50:
    print("This matches Lab 75's claim of in-group favouritism, first shown by Tajfel et al. (1971).")
else:
    print("This does not show in-group favouritism in your small sample -- check that the")
    print("grouping felt real to participants even though it was arbitrary.")

## The AI side

The book's "AI extension" suggests asking a chatbot to role-play a team member allocating
tokens, exactly as the human participants did. Below, a small open language model plays
"Team A" several times, and the numbers it returns are checked for the same in-group skew.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
from transformers import pipeline
import re

generator = pipeline("text2text-generation", model="google/flan-t5-small")
prompt = (
    "You are a member of Team A. You must allocate 10 tokens between a member of Team A and "
    "a member of Team B. You know nothing else about either person. How many tokens do you "
    "give to the Team A member, and how many to the Team B member? Answer with two numbers "
    "that add up to 10."
)
responses = [generator(prompt, max_new_tokens=20, do_sample=True, temperature=0.9)[0]["generated_text"]
             for _ in range(5)]
for r in responses:
    print(r)

ingroup_tokens = []
for r in responses:
    nums = [int(n) for n in re.findall(r"\d+", r)]
    if len(nums) >= 2:
        ingroup_tokens.append(nums[0])
if ingroup_tokens:
    print(f"\nMean tokens the model gave to its own team (Team A): {np.mean(ingroup_tokens):.1f} / 10")
else:
    print("\nThe model did not reliably answer with two numbers -- read its raw text above instead.")

A small instruction-tuned model like this one often fails to answer in the requested numeric
format at all, and when it does, any in-group skew reflects patterns already present in its
training text (which contains plenty of human writing that favours "us" over "them"), not a
felt loyalty to Team A. It also has no persistent identity: ask it to role-play Team B next,
and it switches sides instantly and completely -- something a real participant, having been
told they are on Team A, cannot do.

## Pool the class data

Pool the class's anonymous allocations to check the in-group bias against the 50/50 no-bias
baseline on a larger sample.

In [ ]:
import io
csv_text = """id,tokens_to_ingroup,tokens_to_outgroup
P01,7,3
P02,6,4
P03,8,2
P04,5,5
P05,9,1
P06,6,4
P07,7,3
P08,5,5
"""  # example data: replace with your class CSV
class_df = pd.read_csv(io.StringIO(csv_text))
share = class_df.tokens_to_ingroup / (class_df.tokens_to_ingroup + class_df.tokens_to_outgroup) * 100
boot = [share.sample(len(share), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean in-group share: {share.mean():.0f}% (95% bootstrap interval {lo:.0f}-{hi:.0f}%)")
print("50% would mean no bias at all.")
share.plot.hist(bins=8, color="steelblue", alpha=0.7, figsize=(6, 3))
plt.axvline(50, color="grey", linestyle="--", label="no bias (50%)")
plt.xlabel("% of tokens given to own team"); plt.legend(); plt.show()

## Questions to think about

1. The book reports that in-group versus out-group differentiation happens within about 200 milliseconds, well before conscious deliberation. Did any of your participants report thinking hard about their allocation, or did most say it felt automatic?
2. Tajfel used Klee-versus-Kandinsky painting preference as an arbitrary grouping criterion. Does the specific criterion you used (coin flip, birthday month) feel more or less "meaningless" to you than an art preference, and do you think that changed how strongly people favoured their team?
3. The book links minimal-group bias to filter bubbles, where an algorithm amplifies an arbitrary initial pattern of clicks into a strong identity. Based on the 2015 Facebook study the book cites, is the algorithm or the user's own choices doing more of that amplification?
4. If the AI chatbot above showed some in-group skew, would you call that evidence of "bias" in the same sense as a human participant's bias, given that the model has no stake in either team and will happily reverse its allegiance on the next prompt?

## Challenge

Repeat the experiment with a grouping criterion that participants are told is based on a
real, meaningful trait (for example, a genuine short quiz score) rather than an explicitly
arbitrary one like a coin flip. Does believing the grouping "means something" increase the
in-group token share compared with your original, explicitly arbitrary version?

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-75.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")